## Part 2 — Conditional Edges (Branching)

Fixed edges always go to the same next node. **Conditional edges** let the
graph branch differently based on what's in the state — this is how agents
decide whether to keep working or stop.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1dLbTK0PTwfGKgbX4-IC55DEMfUgvLcND#scrollTo=x0Emx4hHoSa1)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://python.org)
[![LangChain](https://img.shields.io/badge/LangChain-0.3.x-green)](https://python.langchain.com)
[![Gemini](https://img.shields.io/badge/LLM-Gemini%201.5%20Flash-orange)](https://aistudio.google.com)


### Copy the links from below badge to visit my pages


[![GitHub](https://img.shields.io/badge/GitHub-View_Profile-black?logo=github)](https://github.com/mtptisid)
<a href="https://www.linkedin.com/in/siddharamayya-mathapati" target="_blank">
  <img src="https://img.shields.io/badge/LinkedIn-Connect-blue?logo=linkedin" />
</a>
[![Portfolio](https://img.shields.io/badge/Portfolio-Visit-orange?logo=google-chrome)](https://siddharamayya.in)



In [1]:
!pip install -U -q langgraph langchain langchain-community \
            langchain-google-genai langchain-text-splitters \
            google-generativeai wikipedia duckduckgo-search

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 whi

In [2]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_KEYS")

In [3]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

class AgentState(TypedDict):
    question:   str
    answer:     str
    confidence: str    # "high" or "low"
    attempts:   int

def call_llm(state: AgentState) -> dict:
    """Call the LLM and self-assess confidence."""
    prompt = f"""Answer this question: {state['question']}

After your answer, on a new line write exactly:
CONFIDENCE: HIGH   (if you are certain)
CONFIDENCE: LOW    (if you are guessing)"""

    response = llm.invoke([HumanMessage(content=prompt)])
    text = response.content

    # Parse confidence from response
    confidence = "low"
    if "CONFIDENCE: HIGH" in text.upper():
        confidence = "high"
    # Strip the confidence line from the answer
    answer = text.split("CONFIDENCE:")[0].strip()

    return {
        "answer":     answer,
        "confidence": confidence,
        "attempts":   state.get("attempts", 0) + 1
    }

def improve_answer(state: AgentState) -> dict:
    """Retry with more explicit instructions when confidence is low."""
    print(f"  ↻ Low confidence on attempt {state['attempts']} — retrying...")
    prompt = f"""You previously gave a low-confidence answer to: {state['question']}
Try again with more careful reasoning. Be specific and thorough.
After your answer, write CONFIDENCE: HIGH or CONFIDENCE: LOW."""

    response = llm.invoke([HumanMessage(content=prompt)])
    text = response.content
    confidence = "high" if "CONFIDENCE: HIGH" in text.upper() else "low"
    answer = text.split("CONFIDENCE:")[0].strip()
    return {"answer": answer, "confidence": confidence, "attempts": state["attempts"] + 1}

# ── Routing function — decides which node comes next ──────────────────────
def route_on_confidence(state: AgentState) -> Literal["improve", "end"]:
    """
    This function is called after call_llm to decide the next step.
    Return value must match a node name or END.
    """
    if state["confidence"] == "low" and state["attempts"] < 3:
        return "improve"   # → go to improve_answer node
    return "end"           # → go to END

# ── Build graph ────────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("call_llm",      call_llm)
builder.add_node("improve_answer", improve_answer)

builder.set_entry_point("call_llm")

# Conditional edge: after call_llm, call route_on_confidence to decide next step
builder.add_conditional_edges(
    "call_llm",
    route_on_confidence,
    {
        "improve": "improve_answer",   # if function returns "improve" → this node
        "end":     END                 # if function returns "end" → stop
    }
)

# After improving, go back to call_llm to re-assess
builder.add_edge("improve_answer", "call_llm")

graph = builder.compile()

result = graph.invoke({
    "question": "What was the exact GDP of France in 1987?",
    "attempts": 0
})
print(f"Final answer (after {result['attempts']} attempt(s)):")
print(result["answer"])


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Final answer (after 1 attempt(s)):
The exact GDP of France in 1987 was **$989.71 billion** (current US dollars).


**Graph structure:**

```
START
  │
  ▼
call_llm ──→ [route_on_confidence]
                    │
          confidence=HIGH ──────────────────► END
                    │
          confidence=LOW (attempts < 3)
                    │
                    ▼
            improve_answer
                    │
                    └──────────────────────► call_llm (loop)
```

---

## References

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph)
- [LangGraph Tutorials](https://langchain-ai.github.io/langgraph/tutorials)
- [LangGraph How-To Guides](https://langchain-ai.github.io/langgraph/how-tos)
- [LangGraph Conceptual Guides](https://langchain-ai.github.io/langgraph/concepts)

---

*Part of the [LangCraph Tutorial Series](../README.md) — built with LangGraph 0.2.x and Google Gemini 1.5 Flash*
